# v4 full run on a Colab GPU

Runs the v4 pipeline on all 410,201 NEISS narratives and saves everything to your Google Drive, so a disconnect does not lose work.

**Before you start:** Runtime > Change runtime type > **T4 GPU** > Save.

**Drive layout** (made by the first cell): `MyDrive/NEISS-v4/data/` for the input and `MyDrive/NEISS-v4/outputs/` for results. If a teammate shared `neiss2025_clean.csv`, put it (or a shortcut to it) in `NEISS-v4/data/`. Otherwise step 4 builds it from the CPSC website (about 15 minutes).

**If Colab disconnects:** run all cells again. Finished chunks are skipped, so it picks up where it stopped.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
BASE = '/content/drive/MyDrive/NEISS-v4'
DATA = f'{BASE}/data'
OUT = f'{BASE}/outputs'
os.makedirs(DATA, exist_ok=True); os.makedirs(OUT, exist_ok=True)
print(os.listdir(DATA))

In [ ]:
# 2. Get the code (a fresh copy each session; it takes seconds)
BRANCH = 'v4-speed-and-classifier'   # change to 'main' once the v4 PR is merged
!rm -rf /content/repo
!git clone -q --branch {BRANCH} https://github.com/Break-Through-Tech/Marsh-McLennan-1A-object-detection-model.git /content/repo
%cd /content/repo
!pip -q install -r requirements.txt

In [ ]:
# 3. Check the GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

In [ ]:
# 4. Input data: use the Drive copy, or build it once from CPSC and save it to Drive
import os, shutil
os.makedirs('data/raw', exist_ok=True); os.makedirs('data/processed', exist_ok=True)
clean_drive = f'{DATA}/neiss2025_clean.csv'
fmt_drive = f'{DATA}/neiss2025 1(NEISS_FMT).csv'
if not os.path.exists(clean_drive):
    !wget -q -O data/raw/neiss2025.xlsx "https://www.cpsc.gov/cgibin/NEISSQuery/Data/Archived%20Data/2025/neiss2025.xlsx"
    import pandas as pd
    for sheet in ['NEISS_2025', 'NEISS_FMT']:
        pd.read_excel('data/raw/neiss2025.xlsx', sheet_name=sheet).to_csv(f'data/raw/neiss2025 1({sheet}).csv', index=False)
    !cd notebooks && jupyter nbconvert --to notebook --execute milestone1_eda.ipynb --output-dir /tmp --output eda_run --ExecutePreprocessor.timeout=3600
    shutil.copy('data/processed/neiss2025_clean.csv', clean_drive)
    shutil.copy('data/raw/neiss2025 1(NEISS_FMT).csv', fmt_drive)
shutil.copy(clean_drive, 'data/processed/neiss2025_clean.csv')
if os.path.exists(fmt_drive):
    shutil.copy(fmt_drive, 'data/raw/neiss2025 1(NEISS_FMT).csv')
print('Input ready')

In [ ]:
# 5. Benchmark on 1,000 rows first: note the rows/s it prints
!python -m src.extract --sample 1000 --batch-size 64 --output /content/bench.csv

In [ ]:
# 6. Full GLiNER run (object names). Saves a chunk to Drive every 20,000 rows.
!python -m src.extract --batch-size 64 --chunk-size 20000 --output {OUT}/extractions_v4.csv --log {OUT}/v4_full_run.json

In [ ]:
# 7. Product codes from the v4 classifier (about 5 minutes; CPU is fine)
!python -m src.classify --train --model-path {OUT}/product_classifier.pkl
!python -m src.classify --model-path {OUT}/product_classifier.pkl --input {OUT}/extractions_v4.csv --output {OUT}/extractions_v4_coded.csv

Done. `outputs/extractions_v4_coded.csv` has, for every narrative: GLiNER's object (`extracted_object`), the classifier's product (`predicted_code`, `predicted_product`), and `agree` (whether the two point to the same product).

Next: rerun Tasks #11 and #12 on it, e.g. `python -m src.summarize --extractions <path>`.